# 00 — Pre-check del entorno

Valida que el entorno virtual de Python tenga instaladas todas las librerías necesarias para ejecutar el proyecto.

**Resultado esperado:** todas las celdas deben ejecutarse sin errores y los checks deben mostrar ✅.

## 1. Versión de Python

In [ ]:
import sys

major, minor, *_ = sys.version_info
assert (major, minor) >= (3, 9), f"Se requiere Python >= 3.9, se encontró {major}.{minor}"
print(f"✅ Python {major}.{minor}.{sys.version_info.micro}")

## 2. Librerías requeridas

In [ ]:
import importlib
from packaging.version import Version

# (nombre_modulo, version_minima, descripcion)
REQUIRED = [
    ("pyspark",    "3.5.1",  "PySpark"),
    ("dotenv",     "1.0.1",  "python-dotenv"),
    ("yaml",       "6.0.1",  "PyYAML"),
    ("pytest",     "8.1.1",  "pytest"),
    ("pandas",     "2.2.2",  "pandas"),
    ("pyarrow",    "16.0.0", "pyarrow"),
]

failures = []

for module, min_ver, label in REQUIRED:
    try:
        mod = importlib.import_module(module)
        installed = getattr(mod, "__version__", None)
        if installed and Version(installed) < Version(min_ver):
            msg = f"versión {installed} < mínimo {min_ver}"
            failures.append(f"  ❌ {label}: {msg}")
            print(f"❌ {label}: {msg}")
        else:
            ver_str = installed or "versión desconocida"
            print(f"✅ {label} {ver_str}")
    except ImportError:
        failures.append(f"  ❌ {label}: no instalado")
        print(f"❌ {label}: no instalado")

if failures:
    raise EnvironmentError(
        "Las siguientes librerías no están correctamente instaladas:\n"
        + "\n".join(failures)
        + "\n\nEjecuta:  pip install -r requirements.txt"
    )

## 3. Variables de entorno

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path().resolve().parent
env_file = PROJECT_ROOT / ".env"

if env_file.exists():
    load_dotenv(env_file)
    print(f"✅ .env cargado desde {env_file}")
else:
    print(f"⚠️  .env no encontrado en {env_file}")
    print("   Copia .env.example a .env y completa los valores.")

# Solo credenciales: host/port/name viven en config/database.yml
REQUIRED_VARS = ["DB_USER", "DB_PASSWORD"]
missing_vars = [v for v in REQUIRED_VARS if not os.getenv(v)]

if missing_vars:
    print(f"\n⚠️  Variables faltantes en .env: {missing_vars}")
    print("   Los notebooks de ingesta no podrán conectarse a la base de datos.")
else:
    print(f"✅ DB_USER     = {os.getenv('DB_USER')}")
    print(f"✅ DB_PASSWORD = ***")

## 4. Archivos de configuración

In [ ]:
import yaml

CONFIG_DIR = PROJECT_ROOT / "config"

# --- database.yml ---
db_config_path = CONFIG_DIR / "database.yml"
assert db_config_path.exists(), f"❌ No se encontró {db_config_path}"

with open(db_config_path) as f:
    db_cfg = yaml.safe_load(f)["database"]

DB_REQUIRED_KEYS = ["host", "port", "name", "driver"]
missing_db = [k for k in DB_REQUIRED_KEYS if not db_cfg.get(k)]
if missing_db:
    raise ValueError(f"❌ database.yml: faltan claves {missing_db}")

print(f"✅ database.yml")
print(f"     host   : {db_cfg['host']}")
print(f"     port   : {db_cfg['port']}")
print(f"     name   : {db_cfg['name']}")
print(f"     driver : {db_cfg['driver']}")

# --- spark_config.yml ---
spark_config_path = CONFIG_DIR / "spark_config.yml"
assert spark_config_path.exists(), f"❌ No se encontró {spark_config_path}"

with open(spark_config_path) as f:
    spark_cfg = yaml.safe_load(f)["spark"]

SPARK_REQUIRED_KEYS = ["spark.master", "spark.executor.memory", "spark.driver.memory"]
missing_spark = [k for k in SPARK_REQUIRED_KEYS if not spark_cfg.get(k)]
if missing_spark:
    raise ValueError(f"❌ spark_config.yml: faltan claves {missing_spark}")

print(f"\n✅ spark_config.yml")
for k, v in spark_cfg.items():
    print(f"     {k} : {v}")

## 5. Java (requerido por PySpark)

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True, text=True
    )
    # java -version escribe en stderr
    version_line = (result.stderr or result.stdout).splitlines()[0]
    print(f"✅ {version_line}")
except FileNotFoundError:
    raise EnvironmentError(
        "❌ Java no encontrado en PATH.\n"
        "   PySpark requiere Java 8, 11 o 17. Instálalo y agrega JAVA_HOME a tu entorno."
    )

## 6. SparkSession de prueba

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.spark_session import create_spark_session

spark = create_spark_session("precheck")
print(f"✅ SparkSession creada — versión: {spark.version}")
spark.stop()
print("✅ SparkSession cerrada correctamente.")

## 7. Resumen

In [ ]:
print("="*50)
print("  ✅ Pre-check completado exitosamente.")
print("  El entorno esta listo para ejecutar los notebooks.")
print("="*50)